# Nepali (Model L) — corpus and tokenizer, self-contained

Collects the Nepali corpus and trains its tokenizer from scratch. **Nothing to
clone and no GitHub token needed** — the pipeline source is embedded in this
notebook and written to disk before use.

Run this while the Hindi (Model H) work proceeds elsewhere: the two models are
independent by construction — separate corpora, tokenizers, vocabularies — so
there is nothing to synchronise between them.

## Before running: two settings in the right-hand sidebar

1. **Settings → Internet → On** — off by default on Kaggle, and the single
   most common cause of an unexplained failure below.
2. **Settings → Accelerator → None** — this stage is CPU and I/O bound. A GPU
   does nothing for it and spends quota Phase 2 needs.

Then **Run All**. Expect 1–3 hours.

## What it produces

```
/kaggle/working/nepali/
  data/processed/ne_{train,validation,test}.jsonl   the corpus
  tokenizer/ne_*.model  ne_*.vocab                  trained from scratch
  reports/*.json                                    statistics for the report
```

`/kaggle/working` persists between sessions **of this same notebook**, so if a
session ends you can re-run and the build resumes from its checkpoint.


## 1 · Environment


In [ ]:
import os, sys, pathlib, shutil, subprocess

REPO = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle').exists() else pathlib.Path.cwd()
REPO.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))

# Keep the dataset cache off the 20 GB persistent quota, which has to hold the
# corpus and checkpoints. /tmp is wiped between sessions, so a resumed run
# re-downloads - replaceable bandwidth in exchange for non-replaceable quota.
os.environ['HF_HOME'] = '/tmp/hf_cache'

print('work dir :', REPO)
print('cpus     :', os.cpu_count())
with open('/proc/meminfo') as fh:
    print(f'memory   : {int(fh.readline().split()[1]) / 1024**2:.1f} GB')
print(f'disk free: {shutil.disk_usage(REPO).free / 1e9:.1f} GB')

probe = subprocess.run(['curl', '-sS', '-m', '10', '-o', '/dev/null',
                        '-w', '%{http_code}', 'https://huggingface.co'],
                       capture_output=True, text=True)
online = probe.stdout.strip() == '200'
print('internet :', 'ON' if online else 'OFF  <-- enable it in Settings -> Internet')
assert online, 'Enable Internet in the notebook settings panel, then re-run.'


## 2 · Dependencies

No PyTorch: Phase 1 is data and tokenizers only. `sentencepiece` *trains* a
vocabulary from scratch and `datasets` is purely a download client — neither is
a pretrained model, which the assignment forbids.


In [ ]:
%pip install -q sentencepiece datasets pyyaml beautifulsoup4 lxml regex 2>&1 | tail -2
import sentencepiece, datasets
print('sentencepiece', sentencepiece.__version__)
print('datasets     ', datasets.__version__)


## 3 · Write the pipeline modules

The next three cells unpack the pipeline to disk. Each carries its files as a
compressed blob rather than inline source: ~100 KB of Python with Devanagari
literals and nested docstrings would be unreadable pasted into cells, and every
quote would be a chance to corrupt it.

What gets written:

| Module | Responsibility |
|---|---|
| `normalize.py` | Devanagari-aware Unicode normalization |
| `language_id.py` | **Hindi vs Nepali discrimination** — see below |
| `clean.py` | Quality filters + MinHash/LSH deduplication |
| `splits.py` | Deterministic content-hash train/val/test assignment |
| `sources.py` | Hub streaming, scraping, and local/OCR sources |
| `checkpoint.py` | Resume state, so a lost session costs minutes not hours |
| `train_tokenizer.py` | SentencePiece training and the vocabulary sweep |

### Why language identification is the interesting part

Nepali is written in Devanagari — the **same script as Hindi** — so a script
check cannot separate them, and as closely related Indo-Aryan languages they
share much vocabulary. Since Hindi is far more abundant online, Hindi text
leaking into this corpus is the likeliest way to break the independence the
project requires.

The separation therefore rests on grammar, using the highest-frequency words in
any document:

| | Hindi | Nepali |
|---|---|---|
| copula | है / हैं / था | छ / छन् / हो / थियो |
| dative | को | लाई |
| ergative | ने | ले |
| past participle | किया / गया | गरेको / भएको |


In [ ]:
# Pipeline modules
import base64, gzip, json, pathlib

_BLOB = (
    "..."
)

_files = json.loads(gzip.decompress(base64.b64decode(_BLOB)).decode('utf-8'))
for _rel, _content in _files.items():
    _path = REPO / _rel
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_content, encoding='utf-8')
print(f'wrote {len(_files)} file(s)')
for _rel in sorted(_files):
    print('  ', _rel)


In [ ]:
# Pipeline scripts
import base64, gzip, json, pathlib

_BLOB = (
    "H4sIAFDXg2oC/+09DW/bxpJ/ZZ8LI5SfRMtOW/SUqkDapPeKa5IiyXvFwTZYWlzZrClSR1JxfD7fb7/52G+Skpz2HYripUATkbuz"
    "u7Mzs/O1w/uDZlHn67Y5vtzkRZYsqnq9aeL13cFMHHz2l+NNUx9f5uWxLD+I9V17XZVPz8vzA/zvW+wgqlKKIi2vNumVfNII7j+D"
    "v4tCLtqxKKt6lRb5f8uxWBQyLccik9lmXeSLtIVnDfyrjQFk+f5aitvrqpCiaQGWyBv4Ry3TVV5eiYnIqsVmJcu2EWktxTIvWlnL"
    "TKSNaK/lHTys8w9SpGUmbuu8bSVAhO5pfnXdirxsK2yW1zwedpcAs6nEWqY3YiVXVX2HI+ZlJtcS/le2olqq1YgGpo+gz8tUfDGd"
    "vpq01Y0sofGVbFpRyg+yhv/LjOYi6vRWtPIjAChFljc3tLq/N7Cm2Qz/KeAPo1IM4F5MJouqXOZX4lKWV4C9Y/7ZHK/ztSzyUsZ3"
    "6aoQ5+cMDf9MJk21qRey6fRRz7nLZLKuK1p+k67Whcz0jD4TP9VVtlm0Ocyr3pSiTesr2SLy/xfXLBCZJf6kxTezxy0kbWA82chH"
    "raTTaWApy01RaKrE9SzraiWSZLlpN7VMEpGv1lXdwg6WVZviAhtspZ/WV+u0bqR50Nw15t9tvpIK3jptr4v8UgP7CX4ilLcvf3qT"
    "vH3z5r2Y07MIxoU5JckormVTFR9kNIphACTds5MLoMo7wAw0jPOykXUbTcdI55GBMxqZJRSrNM7SNo0X13Jxs66AjvXw35knr9IS"
    "SKseix+IHN+1xFctbCFyWPJrU5VF0lYdkMiMBhr+kPV3hOaxeKF4TT0OuxqW1t2/ZYp7R0TwWr+FKQEBJWqTQiB6jxWIHNg5MTw+"
    "FkWVZskqLfMlrKnTF5nYdH2Hv543TX5V4pj0U6+EfvxcI3QHCAiYtomZShP6YRBBzxCHedPmi8bptGnzAjsROavmNEtNyQm/63Qp"
    "qqsr4JuEfumejWw360S9GpPM4r06p/8yuRRElQmQZxONxOQbQ6jxa2SKdbqQigeZ8H/CdyCxVoC1bIITwh6Mz1jzBrEsNqyBXA3A"
    "56odgaijTDI7A6PME9yUJBmLJW5ri7u0KIAv56bz2/T2he3wN1msv9dNR+6AcZpliZ5RdH6gRcP5wVjU8r82OYjz+ft6g8R7t5Zz"
    "ZKaxuAZ4c1icwrH4z+evfsS5eOfO+cGOkRSx7TXUO2orNO3RiDsH0ER+gMfbMt0ULUDSEvbALkMLLKQQPCs07QhFV5Hq8z8o0kY7"
    "hy3yVd5OYIdoabQWkAh2Dq8BS2bwN3BM1XkGg8paYcScqoIgbR/PCGgcmU8HPghxcPvWzMI+8qZjH4fzQoToU0YdsTzMM33UNm21"
    "bmDrYeagGYBUzMzQe88cNnaTFpNlndJZ1537Eri6b/bT+HTanf1bRU+iuUbFBJSG4KDkzeZRtVoE48YCFZ6sui1RiEB/zZ7CTvZy"
    "k8HqUS8hvUKuUtRPQMaB4oKzEYsU0OHAILUjbcVX08NDmgpilFAYPx5RC1hQM0FioYXsj6jP4x48vQR5ClJBik2DE62Q4mHjW56j"
    "s9msraUCh4ctAkml0HApge+lbY9HjJAfQVA3ey0OllRLFL4TVNju+jnmdJpMp1OXYa/gGG+EktQ4OVl/gJ3MS6uSPoJv7Ek+yfK6"
    "i1MSRftyzgsgvEWLqitKRJgnjCjsCI2IVP+Z+BokFEhAwH0C435z7LQaxX2096aEw7BIL8eC9Y72GsgQSGtVbQAFmfj3qroCSfaC"
    "1G6gR9Cqm039AX4xtf7j1SeRnMGO3qO9BMvpF7RrHQS9S9EmcFAiCKx4LW7kurUbGIspqurpZQGzt61hwz9hEbwP3tRZ1OCRALsl"
    "E1DO/PdGltAOmtPBVftKERKPyJd0Cj6CBcJ5Ase1nzLRF7KQLZpEdzw6coYzWbTCCDZTzdo99UBTSNvFdTjdGvShutSzdlUfqxCh"
    "+GNVCAbxdJ+3aK8gwtheIyukoyV4ShDCBhXIHwhfqIN43qvbRdhS6YBjghGrxaneeizoz63Ozg+slnKht6a9btwW9MC+VuhyG2j1"
    "4kKbayCPGuRkaGUtkGMGjWOq1wYoyi9S+jzFM9LtwE4HpXK+xDPHWnDJvZ77gxFxDAiMl2UVKR8A7T5jHnF+CHLnsBmN9ULmh6Sd"
    "dPCR4JAwwbHBmukCrdTLkbGYXfk1sGqvjVk6Kfhbepn3Dno/A26boP0pVyATQlUNVPiSXBGTPf4wQGMz4SwG7KXIMZcijZORpku2"
    "xaB3YJ1FnvFm+s0tUo+ODO4JCuwXYlYBTpXlBJA9SypyDKnIAcHGlwOgMdYSgAgNqO58zI42a0ldPFOPOUzp670k9yOrOoeZkCUK"
    "bBA11DpqRjPBpFbIMiLoQIT4+/wg/hUEU0S7TY2THDTHCyJXdPrwXEZmamjzkEbMVhLYcPgDTaEGpW7nUQP724rXJI0LsMLUJsaw"
    "oTBjUKESNawBbPiJlZ5E6YpqOP+hHjF4un1Q3RiV0USr6SNN3e89JSqrJAMjYS7uZEsqZlczawDngLMFanDXdbW5uhapBtnR1zYN"
    "Coa0FFKpfhmxDqpT5GOCJ6wEgyKfw7mc8WAGHioaOORKpqBZwGv5EeAXdyAuWlSCc+t20kuxnqdYg3knQcaj3smzatD68dTw5wLn"
    "iQ4xVOAzPfsFeYtgTJow4kJDZIZBDiR/5wbtNW1WsodyIm6vczgP0S+oDC12ZYKAzGvA8G21KTK70JKO/ZwPTsS6shZqdFZxY7FM"
    "i0JUG/RXXeawq3VemEVqSK+4G1kBgLgM5NWlxMUD0tAdtF7LbIbjKIuiWaQ0NzC+eMPTG9oyHGmpgeaECzhJAUgO21d9kMrNCZMb"
    "05QRMinHamVEHLcivU3vqCnpXOi50TDZnDEYw/mkl7A4tSAkpUTtwxwP+8gn/SPmBzJQEuQuej5CRvEbEl8gh2i21paShe4oRTiQ"
    "O/SRiE7iKewmjcZ7kmi7cUTDuc2DwRSv+Y1mdjRPqpW+8v2e+Q1OUuJfbUmC5nPYEDh4UbWw1RFo5IfxKY+h+GDkaWz4B471ew8x"
    "s/EDykV87k6On/bh1gE42m8JLrYVT83s7KPDeLpUNiqvdU6rpdn1L6CzeWq6J9Mp7dTQPvWA6qOoYQAjHugTMMCQZob3xnqxZqka"
    "EaDV9nIks6Jm2TQ0w7ctqHctHs76WvjLdJUhR78nHXIv5WdAG7KwlF6maM57Cmdz5Gt9x4BVx3A159mKXeCofoRu8cgHanVNy540"
    "tjKGSJwpcDEbVtFoiGmtDSWuwO4tn6HE5ZBJn1UEfKw0YW9KDjXpgVFLqyOrKKFLHxbnOPj71Dytbvfp0GxiwTGDc5qL79OikR0E"
    "4CG0c/3Ez3PTBh9EcEjCaQqbtGsKakAFxdFgZj5NNxzEoEhhsk7vsIPSFGXmN3VWhR5d/+UwezKLvtWdyTS1+zIDEr+ik+kQmPYQ"
    "fWyg5jUSBCH+QufB2EpnEmkdvjTriBWsRvxVnLDQ1c9l6UhifoiwOw9Zi6NxQmEUCCTm17cSlIpFriwXOFjXGw56NuI2b69D54Ly"
    "rKGMKtGaA6UohKgYUdzkBao7qzybUMyCdAMgV1AKVggDLaRGRz5hnJQN8yLFKI4P0pnAgp1Kz+wUEERbrdmJWcHByopFVm1A5Z+w"
    "bgiAVyFQ2I5UG6O31NExTxZFmq9EBf3q27yRcSBIKxUeJsN0LDRVk6LJG0ZvgWRWaX2n9GzHCTgW9w+jGJCy8plG/0GDE30OgUzD"
    "bTZ2dnJvp/AQU/Cu45xjPw1qevNunC9aUzBDT37U7Yscz93/MjfteqbrMNBtWqPVGPU30sfdIcb9HVebxh8wCpPFpaJCcZ3i02ei"
    "b2kWYugCAyXxWkJP5dhGhZIOyaZaSSc/4BZIFEgaJMOEwwZOlKDvDwVjectxKwxGLBMyuvo4b4AD8Q+qg9uxqk4RRlytWTZDL6tG"
    "Gc4gmJ8zGXNC6D/KSUCnGfrvyLUR3feJPhQ+5wczYUUR2skofexTlnPnB7X8lVBi3+gn4z7Q2C1RQtEFxc/0MMltVWdhC3oWAH0I"
    "kAv06x8Mw+eIixNyM3BHGjDygPRsYLhLCqmZk8CCdiyGZD4aVcSZJC/LjTLzwLP+c4JlFE+T2oF4WLTREBQ1Hph43gG+XVfBE55U"
    "FWLFsnLZawkyNWMl5Rl7b/lglM11r9Li6CbOCTcXJ9pfJz/kFcxZFum6IZ2B26rfSYMEDwSgwdQtNcKMi3hVwYZWZb7QXlnnADSA"
    "nGcmhUXWE2vbOy4JOjAa49VQFps2tzkshFY8GdyXFaZ2MMRaLhEanCXAlRhyJN1cCVCwiGENbZ0DNmXmGK9NcnmXOBOZC3cvuw3U"
    "KnlaiQpt+nqag3+OYXSVZg6ZaBeWBAqEYzmxDSI6sWt9gM1oUuRPD1hHJRTIukF3EGIIw9E2CEvK2bLYNNd0UrvqBfk3PE+7SyUS"
    "UaEZkrMwSA4FDSk4EzYk0RQ01GJII9jvod+O+uAbYuqO4tJU0InkU38nFmdBJ59uOxRrG26hmmF6cVDraibQ0d/pzmAdaaKYSskg"
    "R/x0RgrYl/SZgNP/KqKQizHZjlm8x9xBMo0UVWnEuoLa+mVJeXVyenz70A0ksCI519bBCBMFGSkzzxQx5oOKWYW4DIQ1gzDnK4uV"
    "qKefdzSj9uvwYZLuou62MzAqpyZZA4S0ny/Vo5iRR7vneHac0eSInhuXdJ/9cpPDCVk1OSXNzdU6ubd53NMP/bwYIczUULpr53lP"
    "36pEM5Ki7mDcri6zVBifPQajedSZGFBGBwaKmSAiA2rU3x1IQg8h5nMxOWHC0L1I0TD2QGeUfpjkGOzHXZwAI7VoMiRJ1LPMnlmG"
    "Dq0+S8NEm1Ay2tCTzdyLNOnE6CzuGSUQwYrPIgu437CgbqCPDehiKsAKmgVa6n1EW3DMRcWgYv5HxIqomfO2TVQcSn9FBG8shro7"
    "z3pla496lmZZ9FhwvZhSS52jnYPHAijC/fhyjxBQrzC2hRMYoN+ew+KsZ1YXvud7Dyhk7vZAGovpCCT+jmkNPHZ8uuY4HhgaLW3d"
    "GPXR6QDIz8QbjFSESVFAk+yDjb0oyY2Ua1BaiuqW0jCbagioFkLo0mjFXS4LOPvI48HuWfJ0WI8Bx0mcsFEXIuuaeQn2Kdu0l5Ic"
    "hhvMPF/Gg+KpG8RwI4EorTpY/Wbe7TUb3v3tXjPf1nsROvl1Yh7mALi7oGw/eMge59EzyubjuBEplluN9J4owKA7rK/fEFk94dk8"
    "QYLaBWs0/KqjuXd9kV5zaHgzIBg6ij5F2DrKAmhUPWrFN/NO/4GNVnKStPhoYGX99kOsVJxoNNDt0erO44nPjQPQNLXXROd1UQ6K"
    "658dbd1aIpHODB0X0A4PbCdmEm5qx+YRh9psR9tT23RzMR3Yr5q19KgH0sSxr0biGBjsY3QiJ/82Fnup4bvkyxZJkRW/g8RWpsBW"
    "SMylu6BwCu5chbcQpkLGGGf6V3yypfd6gXQKmy0iaH3PQOBfxz0oiU+WDxS95N9jHbG/pynw69GQd3OLZ9DOwzegP4VDkBzmiiXm"
    "lJQF08JQo0PL8BzeYAySmOa4eSSXBCGMIRYaBslY7hFvZue6A452TLHDq4jSMXHQbvet/LjA4f9D3l1WaZ39gPm+9WYdcsFngnJL"
    "VeREfszJW4YRWyftgk442K+G02uuKZ+03FcKP1L67nTVnx+YxWCoZZcAjcVbOcGLYGR0G/chZ26TBh/37uz+dDAKI3mUAnrydOoK"
    "0X7PhsWCbmvdjJ/kj9CJceo6UHhepY3nDvGuDnmuE6ehiU3zPZK5uHc9bDYrdGbdFn0NVJrkbEsCpdttVQH9ec3VE7+ZySid6bix"
    "9zp00kIz9chr5qs92Mp/4jXWTmI36sBPvGZMZwRMO2e89/rW0Azw6eQzOgt2khwx2KATFp0WJofxwZ+hviY0E2f3NzPxgdwuN2P4"
    "h8oR1NE9snUxfnZ+ADSFXR5U/FAuTD6hj3Gfwmg63hN/Ji7VE8LcBwE+LC3yIu1v1fDB+M/MNbPIZsUeWyCaWCnwiIJTPxjt6u8t"
    "xXT3H4/2m0QAwnk4crKfvdDGROWdqsxv7ZfR6cgY1ICzj/J1KT/UuAkVUfcmmr4wEUU65VTyhhFvZllnT/D9kwsdktsGiUShsIcx"
    "GaB0Mo2GgGMXBXzsHpe2CcbT5DpBKDbVwp/Bz+S31qOboRzcnj2hjLLmycXZE3Jl+wsK0oQoOd3pjJunVbaEaBVmcqb1NvKNJ/bG"
    "1UXfFF+FN6RYnZtRfhunirH3PXIyB2dg8ZxODw9HFjlhypNJcwmX8DXdoeoGzPpPUDNBVjJzvGddVLcUHcEZuPmMsXhJqcnsIMAb"
    "D2t5/Oa7t0Jfeg1Vu/ODvNSJn8RvY0EXerLNwrsjpr2m7l2oq5pewUkdu2DVuk1Sw1gn1Ja+LDnzUhguetIXOtHqyclJo5Nixian"
    "D3dAjcS0BaO5iSzeoE9MaA4Jjt5t4Z/3oEVxMIcSh2UKsqExyd5w8kYdjrAha7Ok0dnsywHuqKvWSxQBJYeBGwFlySiIeH4jTgZx"
    "9Z32D4t0UVdNg0nr5swjkSRWlN3Zsm0TnIpGpzkWX06dbDzMmUbFbJmXeYOnLOqdoD3liiKsZheLH2X6wUvNXaU3eCWKcqhtkJMU"
    "PX32wiabK0iyJN9Ual36OrsG3RJplgWKbY8DY7YlwW0Ab/iafEZGRY1wgnoO9mKs0hmnfEkIBk9IKUoS9qwmCd4ZShLjXAXLoJHi"
    "3V0DJPESlPaI7xQBNEC/KYXhJM2T23yfahjvydWASJEf0mKD2n9QGsPAw9T1RbGhNCdKj64W6eWmQP22uQVJTgUj3mERjIYKRpzE"
    "4ujoHb45OhJRRfes02IsfvllMqEOv/xC+aPs7IAp5BkOb8ZrFPkRCtzB4CXvI0a8RVpcVXA0X6+A8lTOPUe8geeuZZFNMPsc1obQ"
    "UYwjNHSicg54IylOTtF1VHcv78RS1m0OnA0jbS5/pbcV3gfYlDclyLQJmUgLCU2IfE5xmd/nJfpm1RVaWO8EyLxVZA2GVKmUuE3N"
    "2RdV6V0bNvVEniKwl7wT0I4B6fmM9RR48XetnGB6/WW6uGEr0b2owDrUOpcLV0SM1c2IJYp9WS5ylMqUT8BY59nJj3SdG19hIi/j"
    "zOASkbdHcZAuNe5bH0SRx6OhPq5YBw4Bf28ayf+eXFLJhj9nOY7h0hEccKdTDje26XQyuPb7KT3jvX6riLaqlfLOjJUgUtUtRP0C"
    "kZ1Yfk6In+HlyBnabnCw4Xp4M64umcHtEADnFP6rBMafogTG71ShYnepDZI4eI2k9yqxueqsbuxa8p1QlaVjewwJBWrvC82BBPqE"
    "e818iFPCE59DhltM2jQN8ASzTMuSL74yN9g4Hl9EpqeIyid7VyfAgwHOhS31PJ66tQn+YY5iJ/mVqisgcUhz9u3cMtoDQv9jqohY"
    "LAUqxc7xyCE1wXFcauRR4JDPMV8EVPhNmYNts+ILnJdrvircNwtDMaFa+Im3yYkKxrzTY1+l84tP/ElvmFPWgTmsFssrt6F54V9G"
    "77tcvuNiur1zHhxNu6+dm0NL1Yr7I147t8sZ7um1+U0X1tWpzbu6/YYBNfWvFnxAv8g+fa367wPg+iOqEg6I0MjOZ2ygj/zUO8xZ"
    "0H367hr12mswnVd5QzeL73Xnh1jgcTJUg06npDoW9N9cFRx9Ocq/QpWFkOFr6QhSjMxacaqu5Znbu9re4sKEzYJSxFn3p9vKdGik"
    "K76voKxkJelRBqD3IlQdI42xMddlmpMcUJ3oyvnI+iMQjeqVg7w+xDkHBmcf3etxHuhq92rd3j0zF5+tFRtaLAc77uX3GIvCuyaD"
    "yUJqyja3k8071md1cjP9IL0W/WGeTAI+sG8N6yggeLZoGHTa8JMeGPatZT/Sq9EVUrTNjHboDGM6mDN1FlSIYPFzMhPaCpr8hluR"
    "OpOfYDl7yXb6vDN3xwBg3d8sQZfLGe5l1+338veUnA7Ia7wrPBEQtBXebCcQahjlLSMDhB/5icNbk/s7SMeTpNe4CXyyVsiAwmB/"
    "XARxRs04xFtz9Sto071P6b93sDzXdpaXwGDxOaf/h+/Tj6oWA93rmfvFGtSe+G2Az/a4dMx7ZHyBJtZBqty2qKeFp4tHFewd9si/"
    "c6vl/AA0rwqOHmzLrWaDd2eMK53cxl81jMb54eTLRnz//IcfX75QXtztCVzqmmfRhgw7ts89YeA8V7O9CILNAzkYu3Mr+pZiHEvz"
    "w/jpEp1L8I9TSZ6lZAmN4s8xgmHu6OPb5eCqP3mpZhp+sLcXNsxRB4x04MS842krh5htZd4HtQH6R/Mo7JID4F1XRuRRm39LmDr1"
    "5xJ3yOt1JT7kFHJhQWxO5WcUYbOGGhUSsRo8KKyYuNF497d6aGPXRWJmQrQJZU3xPUUdmYh84uhPQ8OldjdcPQ22Wz3dstndq2h0"
    "rIBGkzC6EUTfTTn/BMZWSj6RkxMvUibuZPpnOBoErE7lvsXuun33d9L4GgfLJLV6LPXgVuRgsNnxxpHPXceb78PI3I5EDZu0UKjU"
    "BpeiOw2ZBVSuBAb6+htgGLXecH5GL8F9i8xhFK1eT3uUjoRmc/H1XJzIyeek3EaXI1D2LmVGEYi+NDM0GMH0lhifw66fT005ydMv"
    "XomfrlNQLk/NNbpWZKxQzJ9+9fkzbIY3JGCrMHQ5MACFAmC+K5AzsJuODwFdLXC2PD20y4rF92aFaYEmNwez0uI2vWsGRnAiEgXW"
    "xmhaKxZQV0FdH2uBwwQwlVudknSrj7w9VueJwwFGW7bNYo5zVHpmhge9FcAz2pnem7TMI1m+wlaA2/HgVqnERGzHxRCDgoj+4Gb3"
    "E4pmY69p/Hl/e7MJIO1hp1JAG7efPg0v6jq/H0Z9ivLpTLOrjtd8ip5MIBLjRgk82JF772pYqbPiZ+6IJ6eBFWpzVzA6LUwMG+YC"
    "CjEOFCrb3Sa+uPaO2k5v763fka3FLL/KQaXtqPjOS7+bvr+TGglD7pAOhIF2wRykzLpjSwq6+x7PwHujNz8Q3Xhwjs3JSYkDzk47"
    "hlzwwu7UaL/bxE5OXhjtcKjHsyi8Ece+c2dM2n2fXm+bdXX7XQhiL4KpnWBzmL4ZUpnRVaOXdvZEsxgPaDKIhm0Cm+1lTmflEuzl"
    "5acz1ynyyUav1OEul5NNDCyiGN7gvDrlfEDw1lyI0MCNtSfDuByUZw8juiaF0za3z1UFQ51JYyO9Xnva40S/vQtG0RFgf0bqoW7L"
    "BKSfzk9ODcL5dsAfLHfUUJapzw9dTOaS29JxWW1M7qfr/fGH580jaPxPm07rJ68GW0RZrHY7/cn624NN3Zi9l/SqNoCyXXXk3svK"
    "ZNUupcIZl1VVBOaLTbfcV/X0Uh15s7ckOVq3pTFgNMGj6ztMb+yVLf/wgzUztTiWtxrtHYnqwzDK2MyTP2jmqrpmx5iq50I0WkQv"
    "wL+rdAxUiGYuwKfSBaJ1pv5StJ00jpme1OGhkx6oYHWN3F6o39EFOqwZSQub2Zkt3ZmFpQiHJrgHKJzYLlDvqGKlqpFPWZFqrSe9"
    "a+UClwyPczEH10vkITBOrw7/mbCQ0QYgPuOUWDyTPNdw76nEU3F4Tp2NTGPOSHS7J55q13aN9Uhg/mvYpbwAu53cgptVdMJu+Y84"
    "vBGvyhEuP2Lun+lZ3QwlxL7FNhNsJDT4GbEBrvEwE1YWjHtmwi5s3Wa0JakwSCPcwv//3Iy2ToHU3Rlt31G+KIUxdBlYyhAmEuYA"
    "AlqTOh8NzKZ1rWgSHfZg4JmvH/18fcelwLGGE9COKslKisR5uVVHwGRHtjdPwpKhlC5JHs8jVfUwXbay9lKwjsZci1WHNS5hPJY1"
    "8FjVR1MJjZTSi2opBpBaVZBxW5FcrF6bN+clI1ks+KIP2ZtVtaZCq/WmbGyCmud6Ih86B5cMMs9LPgO4j4t6rjTBt4918atGtm7S"
    "ql/NRqw2mIW1yNHYRZv5qtHx8uYa3lM92fQqxcQFesofeVIfi8A9QxsYUwzpHJsAkfzyC9WoLbDklqxzqgOg0hpFnlHWbCqWBcxw"
    "k5ftyZfH+NfTU/y81ipdn5dpXad3ej9UCVrOqdEehScNLYzDcTVX8lLlcRHZKW27Jrjz8lrW9O0BTR+AjDtVSZbhOAl5YI/vk2vX"
    "4ZM9c+0eDW1Hjp2D9j9ZKp0apdysAB0psNTaPGNE7vja008//vA+ef381ct3Y865C7PUdqa8ffvmXfLDi7F4qf42ZgfbXfW/0tf+"
    "eOlrjn/Xg747R4n9BkTkO0exPLczge05ikEugWklYNpoCcgyT5DI+2d/VukL9vb1fwLlrXNm0t3NFbIs502ZuuveF4vURRF1NnzK"
    "hzqWeZlZtos8Z8lM8Mbpnce7gDVxBz73+OHHamFyrjrHJ+8q0lHq5F8pqf4WlSI3eGn1o5n4Gd3TZaUhoBZAHykBYVzqCuQutzkZ"
    "HqCAUqE4dkt46xrFV0V1GfnpMkexMqCDfA0LckfKxuuqZ8mqTt+9N36QBNOT3u3kwegPPsql5CDKqmowGWcBGAA9DgbKlzl6hPyz"
    "mtx1UVmVnGY54hnFHmHgrW27PKy9c6fLOIFKtKZcaDg3mjZZ4RG0V0Iea6L6c1qVrzZVy9+Sg4dHTtykS5lQJWMn+w4OuTSjnIEI"
    "7JYKveXATpt2OfnK7udvzcb7nfLnfD0jzJ/7//2AinWuq4wy/naF1XvrUD7skxm3w6uhjnCUYYeNqf+LGqy6JGZS/ezUQsz4Lkil"
    "C3CBa6+THxywKXNVHdMHXNgRhUW2DKexPgzW63KZ8yda8ergl1988fRLbVCjqnqdFvrzWcBjVEGZw09c0VHFT8t1rOCBNHEm8rU4"
    "PTqCp1RuTDV6emo9iSqfCf1pZyBzx0InNd0/BHcFUbo4etYsDDvslSrIN/56ahgrGWhBDSX/daL87+zHe1ecBDimEnUY39fXAS3Y"
    "IBDdV3dsn8wkm1s9R6vcnMf6gy3TnoBQ8ByExlcJ+XX855dADxj2oBQz2LEyI32BE81M3J41y0v0duDO2hQ0k7sDtKRLAnIQw0FC"
    "gFNKeJxjF2Qy8tlchJUtXuKlL1tsEL9gCIImY34C5fkYFGfnaLiG4XVEGC22Tulw9Mc17P6RzTNWL9Tdt5UgG47LcNFBf1uxW6Ah"
    "ww8QYw9lQ0BUdvJMq/FHlv9IUMuIb6VRypfSmkZa2w8WazfXVq4NdljVXINBQ4Kyu63a9NTSc7beaaQn2jlPwpwNs/c96RpMPbqw"
    "IZBP2hD54FTHLC3m9P+RlxFhC2UMh4Kc1mVLZTc6aRGeS99+uDdspjXZmUJnp4HFIgUezK9OQ4tJbGh/9UL00pNcsCCm7FeV3K/E"
    "TONpB1LgivXHfRQkfk+gNK4sTqjCkKZDqsCkfwzC21qX48Hd7q4UoTuPLHl8sPAyeUSN+oeYDKWgTmlB1b6AGrGcOaiAJUbeeLhR"
    "tynoBfS1Mz10mKWH1IeBcBWHxW92zcmpsEeHTOUW8XToV8T8QNXUhzsqIrtQ3x6ieZb0cLQlK8w4fLnyGFM7X/J+lX9LBQr0pPlG"
    "PMGlGIy6JW/HAYyfTE8/Pzo69VhXneHqvjyGFHHW+32Npntr307R/34Q16boKZjm3O43JOqWSHM/LaTx2ZMr2EOxI/8agzlb1YJ1"
    "NJ3rZPJnFpwAe+PWBSMftYWAAQM9F92Yv/vGks0ksTYxhmdlY+oqWS/pnIya6bjvK2wTb8p/8LitzUbQHZCVHOXWa+5m9c0cRdMP"
    "xeqbC7peTxHGX70P0M08dAWzsxvHstH+9hv2fdlu1rM5fb1WnCPl0dk3812dDS0445lnXkvtUzFlRpJqmTDgzrjH+3/sz3xb7DEh"
    "ZlWeev/48stuwGHvuPJ7/6vXbpkZd9HDRT6eA6exq1dFRh0ADikMAih7Pp/mldrRW+MLNvqkRgfhYbmy84NXL99jzvtSYfHMp6gL"
    "3iEA9u5vb96+x+oH94ZAQmDqi2UG0FaauUCDeToahx+V49tTfVPZv6JNsGd0gUlVteE4kPheJdnaIjMgvnP8gI76+iaXuOnWs9GV"
    "boK6NsItY46KyK+bps2Xd0FcSvXi9fWVtXlUqNXng98/zvrwf8tynXZFhwAA"
)

_files = json.loads(gzip.decompress(base64.b64decode(_BLOB)).decode('utf-8'))
for _rel, _content in _files.items():
    _path = REPO / _rel
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_content, encoding='utf-8')
print(f'wrote {len(_files)} file(s)')
for _rel in sorted(_files):
    print('  ', _rel)


In [ ]:
# Nepali configuration
import base64, gzip, json, pathlib

_BLOB = (
    "H4sIAFDXg2oC/7VaYW8bNxL9K0SMwE3glWTZThwFLeAm6bVAcg2aHO5DU8jULiWx2iW3S65lXdH89nszJHdXspO212tg2BLJHQ5n"
    "3ryZ4ebXB0bVstTj3JqlXrlxrWtVaqNGO1mVD2biwZH4VptCiy/e2EKV4ttH4u1aOiVORVoqwrNtI722ZvTBHOFHvN9aUTd2qUvl"
    "ZuLayaouVXEttBN+rcTCtqZQhWhaI1qHD96KG9Xo5Y6nO9nSFCQNkoo2VzzXqNo2XvCWyj0X18u2LK+Fl81KeSc+Xkwmb4RvpDba"
    "rCB3o4wbiXeqVLkXW+3XJPA6y6J2IjxujXB5o2vvxotWl8U8t03dulG9w4k+mFKaVStXaiaM6r/Njaww9E+24QdTkYlm4vUHE0TN"
    "RKFupJEr2WgSUku/drMPRohGbueFbkgam7+QXo4xSHPQK1cONrm7opuidXwy/R/V7K3rRmmJ89K7velgPEeTpV3tz9EAqXkkXpRK"
    "BuutYeK1LQvY2SklykqOSJNRTitGvE41LxgAYmkbsV1LL5TM16KwysF0eRTFx77PLEIcidd2qxpsJk1A27FLRhVbtYCovK2UgXNl"
    "owT0aTyWw2MSkIEbTgJMSFKhXS6bIuiuKgFtAn77k9BHu4WsrdwJKaoWypYEngaiaQO7jMNBpKtkWWIyAoIGK23mOdZCzdPpJI1s"
    "bVPQyAUPyNu0BIDEvyDsm0bmFCa0yeNSeRzEPaaTe+zovFgo8bIzzijqTjbw6tYLHK3RCqqLpdRNkCgrhJInedrkuoCZZClemVWp"
    "3Vp8sWhgGkEwdSeQka+NzjGPbSv36EQ4i70Rkvgx1sfzKi/WerVWzXOOtwT2jIVrvxP5WuUbKOPxwYmVMi1Ctdx1u3b+6owVHD9n"
    "ipiJyejpJNkIgCxlrmh5Pz2ZTNN80dYlVPZqTnzQLznrJHhbz82qkVU/Oe0mC73S/r4Jt6sWtuxnTidDr83BSzMxjU570WjPZiOE"
    "B1yOxHc+wCXwWe+0CHJmmuA/NrOM40FisqCB1WHwWkIPZreKsZwwSz6Ha5ayEZUFMuUCrCnJ24bZMUO86YTTSm6gDMJtA2cBDD76"
    "NuA2uFJvQKsKOCPsY8GiwXKewoaqVvhlchV1tMYxjfqReE8rkvvBmhR/DoLJ/bB8BQW/yG3dlhIwyyk/YGgDbD8C1UF8xCpHONkX"
    "QYDJjlMCu3WsqotAuyLYaT6YCCrMwO8tq4kc0tZzdQuBh4NGyaYfo28EpnlHBAMcaoOctp7XiIrWcxpD2D57wpS1BpeUau7AqQjl"
    "wI/vgEkvGDuOQUEWpDySgX2QUoCekbjy9+eiE3H6kLzqvC5Lkvbx4k2cEmtVFsK2/iQ4lpZx4uRNwOYLfISWGLmFGYJfutwaEyUn"
    "NQtmbGAu9rsjpJCpr1NGvgbRlHZLyVq8DtxHW2fYWiwjRbk9Qj7BE7lEqk7EDIsxMCHNCfJfytU4HTMo82ZCH6V9a8VSbQdsToeS"
    "d48lAFBdISCgniNLh5zJZiSnBcfcQImCfcWMwYTh8WT/DScuKIinTyaXpzxSa2OQV51tm5zKkl9/I392KZO34SQ+97sa7m6NJngH"
    "8L6LeWBoFXE23cxCqJMtcDxacOhvGfKLuLE5Ihifd12EeSx1CBazg3IKSpHWhuKbchN2KgArsicsQhS+XVu4QFULVVCSc4lDFjFo"
    "Ef4NjoiUUZYBHm6rVM2WcYEPQkCGY0jn2opU1Z7jjTWMWD99ErNWF7CoiULCZS88e8aJbrEDNy9hmoXMN328seMC/bp+9IhV6jEh"
    "SxBQsUPyaWBdbFtAXlATUP0ZFRts3HNrhpzdREHdIw0Vd8YTcb0lE+IJquB8ps1gUagdmG6Zh4Mo0B4FSJDInDjYGpyhChcIGTBV"
    "cFchaphMNNptHNBR2FhoCNNufPAXiHnFtkx7M0aRUMpULiYWux+j0UTkZzNAjCCndDXCz/irlzqYq49c1YBTKIK6eqg1G2O3Zkxe"
    "ypKXCAIomsgYOA0eUgVrzFCZ9xiA4348BwhOxCX/ZkSciGkYO5viz0/9c33g0HMxdEAbtfqJi9/UDPATgbL4s6DQjzE5Bzkg0Vca"
    "YXwe4TesdDnfU3hxNYB9zrraqlPjcMnpRb8kNAlxPoQnyt+WeJimj8QVKZNTEYV4ZQwPySyqnQCMYLtRPbElPyRZPdEh01ZKOvQr"
    "Ax+B/Aacxyy4ixWZ6naKVksiQwFLvqZ6YBQSUYA2PKlXJiCCiAUbE0ev+yIPcWxW0JfIJQkMaWa/wO6TECEktlzcgOlFVH3Jw2FL"
    "enAUPdCzNVs7MfblZRo6YO0n3dLE3DxCqfSz2Ohd9hlsXE7+ADiGa8DxCFARkjYjBUcMVD8S36M6SrmPC7Mt0jcIrLIIyRxehVSQ"
    "NsVYksadyvP9hjY2YDzIZYtAs6TVDaOKCvkEAHp2yRkUPL3oEPBLi/ovhD7xUk8EQ1pf6wIcE/i/dxLZstIxDSR56pbqa+1J9VjI"
    "lNSNZaiS2Oxd/T/6TAiFFocN+eBEHF4qxHw7uFOIdoyXCq8fpYiKWyIf6iUQEe8S4qrAfrmFi5aNrQ4gCNoP5wOb3TmCdvBD3JTO"
    "fpQIlgXpWOF261eyFkg4SM2pfaS8qdABkM2TC6hsmYmr786/pgTpuTCU1ImsZdepTS/CtQZ5KO5PXQOCOMkHAtnysVWgDdJ9wpH4"
    "t94AOIWWnCy70iwFIOcEfI7VSLp76Q4qtg3Zw1CaG3QoGT/87urNq9Ss9DB/z60otxIhF3KjCeaB0gQvKzZUTwx6FiojRN5Y57Io"
    "jZ4RW9uCRhawUGp99roekulRpLAqfosGCQJ9TLbgqhLuASYbVUpy6HfItdlVs8NmCY8cQ7vYs4d2/WiQMKkqYB3JXhQaakktVMOF"
    "EIa6vuUzPcij2cCcsb8RH9rJs9PFOPyh39NL+n1e8NDZM/4S5qcXPLTkz8swcQCswTOXQwEsczIdign3HHvtVXh8yk88HfdfzhQ/"
    "fknqp/0GqoalpxdBo5F4BYzuuiwgOFfAVaroq/5BR0otJteMqcXkeOrr/PsazRtty9ji7rWawy7zf7hn8xpV+0G0k5hU4YciKMsy"
    "8RJVUGkl3TnW7QKUxxrbRtLsH/1H4rJIUkAIVf4xPucmMnRoHNbtagUTLWUehwlcTvnwREVPjLtnw4pAllwLnp2eTk5HSSIn1VlI"
    "pnEPeGEOSqEelj4nASVdcXJqLbrThjlliKyKvhDnaw1ZY39KOODbFqlml2TQfWy4b0K7Ti0F++2rL6eThzHe9rk3SWzUL62GH0bi"
    "B6kdV9fOgy24QQqbDNSMCSWmdCrTwgDYgVLzV6k2iPHXc2EWy7FFG8oTvmMksMkbqUvuJgeJmnF14DcXefoPu03q8wWz/Dg9ms4M"
    "ho+MjkIOOvR3RM9xboCc4H+cksAYqXH8eARi+qVV/pjuJ6nYEXxvMOo3nceL873naCKbpIf/VnREp5x90ild2utz3p1ER6qFzMr9"
    "LyVvekuAFbYvppLAQCCB4vhqhDtqLoMGKQ4EX4Kk0s1X4psU+xkRQi/RwCzO7zd1OraH5Bbn22IX8RFo4s2dSJiJ71/88GdI4h6a"
    "CNCf27w5AFyJdFXedVBY/0nn0HuEO68OxpAeplelXczE8ePHANrPDtX+8eeAcejY71tft2zy6/RCpGkN6Y6ECDskTsb+1yPxFkvf"
    "vvwGxG+SgEO96mLJTqVXPVy0sNQRmzW6j6+bnc9KFe4X9gwgoiVnyNqKGr6spiULazdiR6dx4uPl5ATFp+Ab+C7hfXwWSysuQGSd"
    "4C74Qp2EjIBBRW8E6KUK1VYZjpnhAOG+jSoVgJXOdx+JRK+SJ4u/0a8s/65n/a0/HlDA3Jr5Ar7ZpM6mF3vo4W/hjYylUg+KEOHX"
    "VYs+cAkcyPwkLeOmJVV7xe/GClmWjU0Fwl+JleCxOSXJOLlv4TD/p02MNrvx87Yp+y71KFRGXDGdn3Jh9CQUdKFsC9OSh864uHo6"
    "mJ6ILxnCgyTVNTFIf3XfONE5197XbjYGm4/6k41ss+JqYPzw1eTh1fnDq1f84eLh5WkcefYkjbxMa76Zp09Xae4sjnz99M7qyOJw"
    "l93OgTOvGtObIBPHsO6HgU740mkVYYZUtrv30dCTfHk8GLJloYu9kXe1yrUsZ8MxNuZlX/SGqjUUytPzMHLngegJtv7ZZFhZs+ue"
    "9pVtmI1++pS4s4t+5/seOKyppzKJiJcsc8dvli3q0OOjapvF4YyCKK6kWwcCgqPqLiVUfjulaor4i2TgUu7m4Z6RriZGcWUuUY3f"
    "+7p4HKOEV/xu8ZRoNNS/WWEZoXEBKAQM6Fu6VfpH90ZvP0UjFa8tcn295hvG1ELxDZP2KCsgmN5OUqtaW6e9vlEhC9sy3S50JET5"
    "PLtxWWpW998uIbNwi0e1Jp+u4F0alSGNUC9ew1ZJFrE9WvIyaMV350Ztxb9+eE39a/gfBdysclXzGSpP5mydt9VfIZylLN2nGafn"
    "AXXL93yj3Fbj+CY0q9S9sSp+/Om+OOyG78KxvyT7PQCefQKAF/8HAL5/9ebt66v3r3BqWRRiB6ML1J9dkQjc0Xsvwt3XsdccVIfh"
    "zVK6rqL/QSDhs5O7zQTfdpCYdFt07AY96ZbuTBqkZYfC4wX1tklgYxfWO8qmDB5+MQ4MNi5e3OF8W3rrs7YGmrvD9QSrjGHlTpJI"
    "akv4Kgr9keXyP6pJmvD5Ua7W6HvjdSoA+eC3/wItVhmTiiMAAA=="
)

_files = json.loads(gzip.decompress(base64.b64decode(_BLOB)).decode('utf-8'))
for _rel, _content in _files.items():
    _path = REPO / _rel
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_content, encoding='utf-8')
print(f'wrote {len(_files)} file(s)')
for _rel in sorted(_files):
    print('  ', _rel)


### Verify the modules import and the discriminator works

A quick sanity check on the property everything else depends on.


In [ ]:
from lma.data.language_id import IndicLanguageIdentifier
from lma.data.normalize import IndicNormalizer, get_profile
from lma.data.clean import DocumentCleaner, CleanerConfig

ident = IndicLanguageIdentifier()
hindi = ('भारत दक्षिण एशिया में स्थित एक विशाल देश है और इसकी राजधानी नई दिल्ली है। '
         'यहाँ हिन्दी भाषा बोली जाती है और गंगा नदी को पवित्र माना जाता है। '
         'उन्होंने कहा कि यह काम किया गया था और लोग इसे पसंद करते हैं।')
nepali = ('नेपाल दक्षिण एसियामा रहेको एउटा सानो देश हो र यसको राजधानी काठमाडौं हो। '
          'यहाँ नेपाली भाषा बोलिन्छ र सगरमाथा संसारकै अग्लो हिमाल हो। '
          'उनले भनेका छन् कि यो काम गरेको छ र धेरै मानिसहरू कृषिमा निर्भर छन्।')

for name, text in [('Hindi', hindi), ('Nepali', nepali)]:
    v = ident.identify(text)
    print(f'{name:7s} -> {v.language:8s} hi={v.hindi_score:.3f} ne={v.nepali_score:.3f}')

assert ident.accepts(nepali, 'ne'), 'Nepali not recognised'
assert not ident.accepts(hindi, 'ne'), 'Hindi would contaminate the Nepali corpus'
print('\nHindi/Nepali separation verified.')


## 4 · Scale the sources up

Widens the Sangraha shard pattern from one shard to all 25 and drops the
per-source document caps.

**The resource gap is visible right here:** Sangraha carries **25 verified
Nepali shards against 100 for Hindi** — a 4× difference in the largest open
corpus for each language, before any filtering. That gap is the phenomenon the
project studies, so report it rather than work around it.


In [ ]:
import yaml

manifest = REPO / 'nepali/configs/sources.yaml'
cfg = yaml.safe_load(manifest.read_text())
for src in cfg['sources']:
    if src['source_id'] == 'sangraha_ne':
        src['data_files'] = 'verified/nep/*.parquet'
        src.pop('limit', None)
    elif src['source_id'] == 'wikipedia_ne':
        src.pop('limit', None)
manifest.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

for s in cfg['sources']:
    if s.get('enabled', True):
        print(f"  {s['source_id']:30s} {s['collection']:10s} "
              f"{s.get('data_files') or s.get('config') or s.get('path', '')}")


## 5 · Build the corpus

**The long cell — 1 to 3 hours.**

Nepali is the lower-resource language, so this exhausts its sources rather than
reaching the 500M-token target. That is the expected result and the assignment
permits it explicitly: *"If the lower-resource language cannot reach 500M from
public sources, collect the maximum feasible amount, report the exact token
count, and justify the shortfall."*

### If the session ends first

Re-run **this same cell**. `--resume` reconciles the output files against the
checkpoint, restores the deduplication index, and skips what each source already
yielded. A resumed run produces a byte-identical corpus to an uninterrupted one
— verified by test, including after `kill -9`.

For an unattended run use **Save Version → Save & Run All (Commit)**; an
interactive session stops when you close the browser.


In [ ]:
def run(script, *extra):
    """Run a pipeline script, streaming output live rather than buffering it."""
    cmd = [sys.executable, str(REPO / 'scripts' / script), *extra]
    print(' '.join(cmd), '\n')
    proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    return proc.wait()

rc = run('build_corpus.py',
         '--config',  str(REPO / 'nepali/configs/pipeline.yaml'),
         '--sources', str(REPO / 'nepali/configs/sources.yaml'),
         '--profile', 'full',
         '--target-tokens', '500000000',
         '--checkpoint-dir', str(REPO / 'checkpoints_ne'),
         '--checkpoint-every', '25000',
         '--resume',
         '--report-every', '20000')
print('\nexit code:', rc)


### Progress

Safe to run in a separate cell while the build is going, or in a new session to
see how far the previous one got.


In [ ]:
import json

state = REPO / 'checkpoints_ne' / 'ne_ingest_state.json'
if state.exists():
    st = json.load(open(state))
    print(f"segment     : {st['segments']}")
    print(f"docs seen   : {st['seen']:,}")
    print(f"docs kept   : {st['kept']:,}")
    print(f"train chars : {st['train_chars']:,}")
    print(f"elapsed     : {st['elapsed_seconds'] / 3600:.1f} h")
else:
    print('No checkpoint yet - the run has not saved one, or it finished and cleared it.')


## 6 · Train the tokenizer and count tokens

Trained from scratch on the Nepali training split only — no pretrained
tokenizer, as the assignment requires.

The vocabulary size is **chosen empirically, not asserted**: candidates are
trained across sizes and both algorithms, each measured on the same held-out
text, and the winner is the lowest fertility subject to an unknown-rate ceiling.
Near-ties resolve toward the smaller vocabulary, because embedding parameters
scale with vocabulary size and Phase 2 has a fixed ~25M-parameter budget.


In [ ]:
rc = run('train_tokenizer.py',
         '--config', str(REPO / 'nepali/configs/pipeline.yaml'),
         '--profile', 'full', '--sweep', '--use-sweep-best')
print('\nexit code:', rc)

# Exact token counts - the number the report must cite, measurable only once
# the tokenizer exists.
rc = run('tokenize_corpus.py', '--config', str(REPO / 'nepali/configs/pipeline.yaml'))
print('\nexit code:', rc)


## 7 · Results

Everything the Phase 1 report needs from the Nepali half.


In [ ]:
reports = REPO / 'nepali/reports'

tokens_file = reports / 'token_counts.json'
if tokens_file.exists():
    t = json.load(open(tokens_file))
    print('=== TOKEN COUNTS ===')
    print(f"  training tokens : {t['train_tokens']:,}")
    print(f"  all splits      : {t['total_tokens']:,}")
    print(f"  vs 500M target  : {100 * t['achieved_fraction_of_target']:.2f}%")
    if not t['target_met']:
        print(f"  shortfall       : {t['shortfall_tokens']:,} (report and justify this)")

stats_file = reports / 'corpus_stats.json'
if stats_file.exists():
    d = json.load(open(stats_file))
    print('\n=== CORPUS ===')
    print(f"  documents : {d['totals']['documents']:,}")
    print(f"  words     : {d['totals']['words']:,}")
    share = d['collection_split']['manual_word_fraction']
    verdict = 'OK' if share >= 0.20 else 'BELOW THE 20% REQUIREMENT'
    print(f"  manual    : {100 * share:.1f}%  [{verdict}]")
    print('\n  by source:')
    for src, words in sorted(d['by_source']['words'].items(), key=lambda kv: -kv[1]):
        print(f'    {src:32s} {words:>12,} words')

tok_file = reports / 'tokenizer_stats.json'
if tok_file.exists():
    m = json.load(open(tok_file))['metrics']
    print('\n=== TOKENIZER (held-out) ===')
    print(f"  algorithm       : {m['model_type']}")
    print(f"  vocabulary      : {m['vocab_size']:,}")
    print(f"  fertility       : {m['fertility']:.3f} tokens/word")
    print(f"  unknown rate    : {m['unk_rate']:.2e}")
    print(f"  byte-fallback   : {100 * m['byte_fallback_rate']:.4f}%")
    print(f"  chars per token : {m['chars_per_token']:.3f}")


### Corpus purity

The check that matters for this language pair: how much Hindi, if any, survived
into the Nepali corpus. Both are Devanagari, so this is the contamination path
a script check could never catch.


In [ ]:
from lma.data.splits import iter_jsonl

train = REPO / 'nepali/data/processed/ne_train.jsonl'
if train.exists():
    counts = {}
    for i, row in enumerate(iter_jsonl(train)):
        if i >= 3000:
            break
        lang = ident.identify(row['text']).language
        counts[lang] = counts.get(lang, 0) + 1
    total = sum(counts.values()) or 1
    print('verdicts on a 3,000-document sample:', counts)
    print(f"  identified as Nepali : {100 * counts.get('ne', 0) / total:5.1f}%")
    print(f"  Hindi contamination  : {100 * counts.get('hi', 0) / total:5.1f}%")
    print(f"  unknown              : {100 * counts.get('unknown', 0) / total:5.1f}%")
else:
    print('corpus not built yet')


## 8 · Download the results

The cell below gathers the small artifacts — tokenizer, vocabulary, statistics
— into one folder. Commit a version (**Save Version → Save & Run All**) and
they appear under the notebook's **Output** tab, ready to download and add to
your repository.

The corpus files themselves are large and belong on Drive or as a Kaggle
Dataset (Output tab → New Dataset), with the link in your README — that is what
the assignment asks for.


In [ ]:
OUT = REPO / 'nepali_artifacts'
OUT.mkdir(exist_ok=True)
for sub in ['tokenizer', 'reports', 'configs']:
    src = REPO / 'nepali' / sub
    if src.exists():
        shutil.copytree(src, OUT / sub, dirs_exist_ok=True)

total_kb = 0
for path in sorted(OUT.rglob('*')):
    if path.is_file():
        kb = path.stat().st_size / 1024
        total_kb += kb
        print(f'{kb:9.1f} KB  {path.relative_to(OUT)}')
print(f'{total_kb / 1024:9.1f} MB  TOTAL')

print('\nCorpus files (large - Drive or Kaggle Dataset, not git):')
proc = REPO / 'nepali/data/processed'
if proc.exists():
    for f in sorted(proc.glob('*.jsonl')):
        print(f'  {f.stat().st_size / 1e6:8.1f} MB  {f.name}')

print(f'\n/kaggle/working free: {shutil.disk_usage(REPO).free / 1e9:.1f} GB')
